# Slopes & Areas — the Calculus of Motion

A **velocity–time graph** packs two stories into one curve:

- the **slope** of the graph is the **acceleration**  $a = \dfrac{\Delta v}{\Delta t}$  (a *derivative*);
- the **area** under the graph is the **distance travelled**  $s$  (an *integral*).

This notebook builds both ideas from scratch. In every demo a slider changes the size of the step  $\Delta t$:

- **Area demos** — cover the graph with rectangles of width  $\Delta t$. With big chunky steps the total area is only an estimate; slide  $\Delta t$ down and the rectangles melt into the curve, so the estimate becomes the *exact* area.
- **Slope demos** — draw a chord between two points  $\Delta t$  apart. Big steps give a rough slope; slide  $\Delta t$ down and the chord becomes the *tangent*, whose slope is the instantaneous acceleration.

Each idea is done twice: on a **linear** graph  $v = 2 + 3t$  and on a **curved** graph  $v = t^2$  (a quadratic "bow" that just rises).

Run the setup cell, then the two import cells, then work through the four demos. Move a slider and the plot redraws live.


In [17]:
import numpy as np
import matplotlib.pyplot as plt
try:
    import ipywidgets as widgets
except ModuleNotFoundError:
    %pip install -q ipywidgets
    import ipywidgets as widgets
from IPython.display import display

# A step-size slider we reuse for every demo: log-spaced so you can drag
# from big chunky steps down to tiny (nearly invisible) ones smoothly.
def make_dt_slider(value=1.5, max_exponent=0.8):
    return widgets.FloatLogSlider(
        value=value, base=10, min=-2, max=max_exponent, step=0.02,
        description="\u0394t (s):", continuous_update=True,
        layout=widgets.Layout(width="520px"),
    )


---
## Part 1 — Area under a *linear* velocity–time graph

$v(t) = 2 + 3t$ is a straight line (constant acceleration $a = 3\ \mathrm{m\,s^{-2}}$).

Cover the graph with **rectangles** of width $\Delta t$; each rectangle's height is the velocity at the *left* edge of its step. The total area of the rectangles estimates the **distance travelled**.

Drag $\Delta t$ from a big chunky step down to a tiny one and watch the estimate converge on the exact area $s = ut + \tfrac{1}{2}at^2 = 66\ \mathrm{m}$.


In [2]:
t_max = 6.0

def v_linear(t):
    return 2 + 3 * t

def exact_area_linear():
    # s = u*t + 1/2 * a * t^2  with u = 2, a = 3
    return 2 * t_max + 0.5 * 3 * t_max ** 2

dt_area_linear = make_dt_slider(value=1.5, max_exponent=0.8)

def make_area_linear(dt):
    fig, ax = plt.subplots(figsize=(8.4, 4.8))

    t = np.linspace(0, t_max, 400)

    # Left Riemann rectangles: width dt (last one trimmed to fit exactly).
    edges = np.append(np.arange(0, t_max, dt), t_max)
    heights = v_linear(edges[:-1])
    widths = np.diff(edges)
    approx = float(np.sum(heights * widths))
    exact = exact_area_linear()
    err = 100 * (approx - exact) / exact

    ax.fill_between(t, v_linear(t), color="tab:red", alpha=0.10)
    ax.bar(edges[:-1], heights, width=widths, align="edge",
           color="tab:blue", alpha=0.45, edgecolor="tab:blue", linewidth=0.6)
    ax.plot(t, v_linear(t), color="tab:red", lw=2.5, label="$v(t) = 2 + 3t$")

    ax.set_title("Area under a linear velocity\u2013time graph  =  distance travelled")
    ax.set_xlabel("Time, t (s)")
    ax.set_ylabel("Velocity, v (m/s)")
    ax.set_xlim(0, t_max)
    ax.set_ylim(0, v_linear(t_max) * 1.15)
    ax.grid(alpha=0.3)
    ax.legend(loc="upper left")

    fig.text(0.5, 0.01,
        f"\u0394t = {dt:.3g} s   |   {len(heights)} rectangles   |   "
        f"approximate area = {approx:.3f} m   |   exact area = {exact:.3f} m   |   error = {err:+.2f}%",
        ha="center", va="bottom", fontsize=11,
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.9))

    plt.show()
    plt.close(fig)

out1 = widgets.interactive_output(make_area_linear, {"dt": dt_area_linear})
display(widgets.VBox([dt_area_linear, out1]))


---
## Part 2 — Area under a *curved* velocity–time graph

Now $v(t) = t^2$, a quadratic bow that just rises. The area is no longer a simple triangle or trapezium, so slicing it into rectangles is genuinely useful.

As $\Delta t \to 0$ the rectangles shrink and their total area approaches the exact value $\int_0^4 t^2\,dt = \tfrac{4^3}{3} = 21.33\ \mathrm{m}$.


In [3]:
t_max_curve = 4.0

def v_curved(t):
    return t ** 2

def exact_area_curved():
    return t_max_curve ** 3 / 3.0

dt_area_curved = make_dt_slider(value=1.2, max_exponent=0.6)

def make_area_curved(dt):
    fig, ax = plt.subplots(figsize=(8.4, 4.8))

    t = np.linspace(0, t_max_curve, 400)

    edges = np.append(np.arange(0, t_max_curve, dt), t_max_curve)
    heights = v_curved(edges[:-1])
    widths = np.diff(edges)
    approx = float(np.sum(heights * widths))
    exact = exact_area_curved()
    err = 100 * (approx - exact) / exact

    ax.fill_between(t, v_curved(t), color="tab:red", alpha=0.10)
    ax.bar(edges[:-1], heights, width=widths, align="edge",
           color="tab:blue", alpha=0.45, edgecolor="tab:blue", linewidth=0.6)
    ax.plot(t, v_curved(t), color="tab:red", lw=2.5, label="$v(t) = t^2$")

    ax.set_title("Area under a curved velocity\u2013time graph  =  distance travelled")
    ax.set_xlabel("Time, t (s)")
    ax.set_ylabel("Velocity, v (m/s)")
    ax.set_xlim(0, t_max_curve)
    ax.set_ylim(0, v_curved(t_max_curve) * 1.15)
    ax.grid(alpha=0.3)
    ax.legend(loc="upper left")

    fig.text(0.5, 0.01,
        f"\u0394t = {dt:.3g} s   |   {len(heights)} rectangles   |   "
        f"approximate area = {approx:.3f} m   |   exact area = {exact:.3f} m   |   error = {err:+.2f}%",
        ha="center", va="bottom", fontsize=11,
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.9))

    plt.show()
    plt.close(fig)

out2 = widgets.interactive_output(make_area_curved, {"dt": dt_area_curved})
display(widgets.VBox([dt_area_curved, out2]))


---
## Part 3 — Slope of a *linear* graph

The slope of a velocity–time graph is the **acceleration**: $a = \dfrac{\Delta v}{\Delta t}$.

Here $v(t) = 2 + 3t$ is a straight line, so the chord between any two points **is** the line itself. Drag $\Delta t$ and move the point $t_0$ — the slope always comes out as $a = 3\ \mathrm{m\,s^{-2}}$, no matter how big or small the step.


In [ ]:
a_lin = 3.0

t0_slope_linear = widgets.FloatSlider(
    value=2.0, min=0.5, max=5.0, step=0.1,
    description="t\u2080 (s):", continuous_update=True,
    layout=widgets.Layout(width="520px"),
)
dt_slope_linear = make_dt_slider(value=1.0, max_exponent=0.6)

def make_slope_linear(t0, dt):
    fig, ax = plt.subplots(figsize=(8.4, 4.8))

    t = np.linspace(0, 6, 400)
    v0 = v_linear(t0)
    v1 = v_linear(t0 + dt)
    slope = (v1 - v0) / dt

    ax.plot(t, v_linear(t), color="tab:red", lw=2.5, label="$v(t) = 2 + 3t$")

    # Chord through the two points (coincides with the line itself).
    tt = np.linspace(t0 - 1.2, t0 + dt + 1.2, 20)
    ax.plot(tt, v_linear(tt), color="tab:blue", lw=3.0, ls="--",
            label="chord (slope = \u0394v/\u0394t)")

    # Slope triangle: rise over run.
    ax.plot([t0, t0 + dt], [v0, v0], color="k", lw=1.3)
    ax.plot([t0 + dt, t0 + dt], [v0, v1], color="k", lw=1.3)
    ax.text(t0 + dt / 2, v0, "\u0394t", ha="center", va="top", fontsize=11)
    ax.text(t0 + dt, (v0 + v1) / 2, "\u0394v", ha="left", va="center", fontsize=11)

    ax.plot([t0], [v0], "o", color="k", ms=7)
    ax.plot([t0 + dt], [v1], "o", color="k", ms=7)

    ax.set_title("Slope of a linear velocity\u2013time graph  =  acceleration")
    ax.set_xlabel("Time, t (s)")
    ax.set_ylabel("Velocity, v (m/s)")
    ax.set_xlim(0, 6)
    ax.set_ylim(0, v_linear(6) * 1.15)
    ax.grid(alpha=0.3)
    ax.legend(loc="upper left")

    fig.text(0.5, 0.01,
        f"\u0394t = {dt:.3g} s   |   \u0394v = {v1 - v0:.3f} m/s   |   "
        f"slope = \u0394v/\u0394t = {slope:.3f} m/s\u00b2  (constant, = a)",
        ha="center", va="bottom", fontsize=11,
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.9))

    plt.show()
    plt.close(fig)

out3 = widgets.interactive_output(make_slope_linear, {"t0": t0_slope_linear, "dt": dt_slope_linear})
display(widgets.VBox([t0_slope_linear, dt_slope_linear, out3]))


---
## Part 4 — Slope of a *curved* graph

Now $v(t) = t^2$. The slope is no longer constant: the steeper the bow, the larger the acceleration.

The chord between $t_0$ and $t_0 + \Delta t$ has slope

$$\frac{\Delta v}{\Delta t} = \frac{(t_0 + \Delta t)^2 - t_0^2}{\Delta t} = 2t_0 + \Delta t .$$

As $\Delta t \to 0$ the chord straightens into the **tangent**, and the slope tends to $2t_0$ — the instantaneous acceleration $a(t) = \dfrac{dv}{dt} = 2t$. Move $t_0$ and shrink $\Delta t$ to see it.


In [ ]:
t0_slope_curved = widgets.FloatSlider(
    value=2.0, min=0.5, max=3.5, step=0.1,
    description="t\u2080 (s):", continuous_update=True,
    layout=widgets.Layout(width="520px"),
)
dt_slope_curved = make_dt_slider(value=0.8, max_exponent=0.4)

def make_slope_curved(t0, dt):
    fig, ax = plt.subplots(figsize=(8.4, 4.8))

    t = np.linspace(0, 4, 400)
    v0 = t0 ** 2
    v1 = (t0 + dt) ** 2
    slope = (v1 - v0) / dt          # = 2*t0 + dt
    tangent_slope = 2 * t0          # exact derivative dv/dt at t0

    ax.plot(t, v_curved(t), color="tab:red", lw=2.5, label="$v(t) = t^2$")

    # Chord through the two points.
    tt = np.linspace(t0 - 0.9, t0 + dt + 0.9, 20)
    ax.plot(tt, v0 + slope * (tt - t0), color="tab:blue", lw=2.0, ls="--",
            label="chord (slope = \u0394v/\u0394t)")

    # Tangent line with the exact instantaneous slope.
    tt = np.linspace(max(0.0, t0 - 0.9), min(4.0, t0 + 0.9), 20)
    ax.plot(tt, v0 + tangent_slope * (tt - t0), color="tab:green", lw=2.5,
            label="tangent (slope = dv/dt)")

    # Slope triangle: rise over run.
    ax.plot([t0, t0 + dt], [v0, v0], color="k", lw=1.3)
    ax.plot([t0 + dt, t0 + dt], [v0, v1], color="k", lw=1.3)
    ax.text(t0 + dt / 2, v0, "\u0394t", ha="center", va="top", fontsize=11)
    ax.text(t0 + dt, (v0 + v1) / 2, "\u0394v", ha="left", va="center", fontsize=11)

    ax.plot([t0], [v0], "o", color="k", ms=7)
    ax.plot([t0 + dt], [v1], "o", color="k", ms=7)

    ax.set_title("Slope of a curved velocity\u2013time graph  =  acceleration")
    ax.set_xlabel("Time, t (s)")
    ax.set_ylabel("Velocity, v (m/s)")
    ax.set_xlim(0, 4)
    ax.set_ylim(0, v_curved(4) * 1.15)
    ax.grid(alpha=0.3)
    ax.legend(loc="upper left")

    fig.text(0.5, 0.01,
        f"t\u2080 = {t0:.2f} s   |   \u0394t = {dt:.3g} s   |   "
        f"chord slope = {slope:.3f} m/s\u00b2   \u2192   tangent slope = {tangent_slope:.3f} m/s\u00b2",
        ha="center", va="bottom", fontsize=11,
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.9))

    plt.show()
    plt.close(fig)

out4 = widgets.interactive_output(make_slope_curved, {"t0": t0_slope_curved, "dt": dt_slope_curved})
display(widgets.VBox([t0_slope_curved, dt_slope_curved, out4]))


---
## Take-away

| idea | what it is | linear graph $v = 2 + 3t$ | curved graph $v = t^2$ |
| --- | --- | --- | --- |
| **area** under $v$–$t$ | distance travelled | $s = ut + \tfrac12 at^2$ | $s = \int v\,dt$ |
| **slope** of $v$–$t$ | acceleration | $a = 3$ (constant) | $a = 2t$ (varies) |

Big $\Delta t$ gives a rough, "chunky" answer; as $\Delta t \to 0$ the approximation becomes exact. That limit **is** the derivative (for slope) and the integral (for area).
